# Cleaning riesgo natural

In [10]:
import geopandas as gpd

# Read the xlsx file into a pandas DataFrame
import pandas as pd
riesgo = gpd.read_file('../../../0_DATA/0_RAW/RIESGO_NATURAL/sintesis_riesgos.shp')  # Skip the first row, use the second row as column headers


In [11]:
riesgo

,OBJECTID,R_G1,R_H1,R_H2,R_H3,R_G2,SUMATORIA,geometry
0,1,None,None,None,None,Inestabilidad de laderas,1,"MULTIPOLYGON (((481202.577 2115603.035, 481202..."
1,2,None,None,None,Sequia,None,1,"MULTIPOLYGON (((493721.142 2106741, 493721.142..."
2,3,None,None,None,Sequia,Inestabilidad de laderas,2,"MULTIPOLYGON (((486401.142 2164805.49, 486401...."
3,4,None,None,None,Sequia,Inestabilidad de laderas,3,"MULTIPOLYGON (((500901.142 2119801, 500896.694..."
4,5,None,None,Olas de calor,None,None,1,"MULTIPOLYGON (((500841.142 2119421, 500841.142..."
5,6,None,None,Olas de calor,None,Inestabilidad de laderas,2,"MULTIPOLYGON (((485499.939 2162325.113, 485499..."
6,7,None,None,Olas de calor,None,Inestabilidad de laderas,3,"MULTIPOLYGON (((486979.716 2161142.902, 486980..."
7,8,None,None,Olas de calor,Sequia,None,2,"MULTIPOLYGON (((486108.391 2164661, 486101.142..."
8,9,None,None,Olas de calor,Sequia,None,3,"MULTIPOLYGON (((492436.716 2120782.698, 492438..."
9,10,None,None,Olas de calor,Sequia,Inestabilidad de laderas,4,"MULTIPOLYGON (((492338.572 2119921, 492321.142..."


In [12]:
# Remove the OBJECTID column if it exists
if 'OBJECTID' in riesgo.columns:
    riesgo = riesgo.drop(columns=['OBJECTID'])

# Replace None with 0, else 1 for the specified columns
columns_to_clean = ['R_G1', 'R_H1', 'R_H2', 'R_H3', 'R_G2']
for col in columns_to_clean:
    if col in riesgo.columns:
        riesgo[col] = riesgo[col].apply(lambda x: 0 if x is None else 1)

# Rename columns to lowercase and update specific column names
riesgo = riesgo.rename(columns={
    'R_G1': 'sismicidad',
    'R_H1': 'inundacion',
    'R_H2': 'ola_calor',
    'R_H3': 'sequia',
    'R_G2': 'inestabilidad_ladera'
})

# Ensure all column names are lowercase
riesgo.columns = [col.lower() for col in riesgo.columns]

# Rename the 'geometry' column to 'geom'
if 'geometry' in riesgo.columns:
    riesgo = riesgo.rename(columns={'geometry': 'geom'})

# Display the updated GeoDataFrame
riesgo

,sismicidad,inundacion,ola_calor,sequia,inestabilidad_ladera,sumatoria,geom
0,0,0,0,0,1,1,"MULTIPOLYGON (((481202.577 2115603.035, 481202..."
1,0,0,0,1,0,1,"MULTIPOLYGON (((493721.142 2106741, 493721.142..."
2,0,0,0,1,1,2,"MULTIPOLYGON (((486401.142 2164805.49, 486401...."
3,0,0,0,1,1,3,"MULTIPOLYGON (((500901.142 2119801, 500896.694..."
4,0,0,1,0,0,1,"MULTIPOLYGON (((500841.142 2119421, 500841.142..."
5,0,0,1,0,1,2,"MULTIPOLYGON (((485499.939 2162325.113, 485499..."
6,0,0,1,0,1,3,"MULTIPOLYGON (((486979.716 2161142.902, 486980..."
7,0,0,1,1,0,2,"MULTIPOLYGON (((486108.391 2164661, 486101.142..."
8,0,0,1,1,0,3,"MULTIPOLYGON (((492436.716 2120782.698, 492438..."
9,0,0,1,1,1,4,"MULTIPOLYGON (((492338.572 2119921, 492321.142..."


In [13]:
import os

# Define the output directory
output_dir = '../../../0_DATA/1_CLEAN/RIESGO_NATURAL/'

# Create the directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# --- 1. Save the GeoDataFrame as a Shapefile (.shp) ---
# Note: Shapefiles truncate column names to 10 characters.
# 'tipo_impacto' will be saved as 'tipo_impa'.
points_filepath = os.path.join(output_dir, 'riesgo.parquet')
riesgo.to_parquet(points_filepath, index=False, engine='pyarrow')